# Try it: auditing a shifted answer sheet

This notebook uses a small synthetic example to show how a skipped bubble row
can make subsequent answers appear to be marked against the wrong questions.

The audit compares possible row alignments and applies a correction only when
the evidence is strong enough. It also checks that ordinary sheets are left
unchanged.

We examine three sheets: a clean sheet, a sheet with a skipped row, and a
sheet containing random guesses.

The notebook should run in about a minute. All sheets in this example are
synthetic; the results should not be interpreted as a live-paper benchmark.


## Setup


In [ ]:
%pip install --quiet git+https://github.com/Enayat-Hassani/omr-registration-audit.git


In [ ]:
import random
from omr_registration_audit import adjudicate_sheet

OPTIONS = "ABCD"
N = 46                      # a 46-question paper, the length of the case study
rng = random.Random(20260101)

# The answer key, and how a candidate who knows about 85% of the material
# would fill the sheet in if nothing went wrong.
KEY  = [rng.choice(OPTIONS) for _ in range(N)]
WELL = [k if rng.random() < 0.85 else rng.choice([o for o in OPTIONS if o != k])
        for k in KEY]

def show(marks, first=14):
    """Print the first few rows beside the key, so a shift is visible."""
    print("question :", " ".join(f"{i+1:>2}" for i in range(first)))
    print("key      :", " ".join(f" {k}" for k in KEY[:first]))
    print("row mark :", " ".join(f" {m}" if m else " ·" for m in marks[:first]))
    hit = ["^^" if i < len(marks) and marks[i] == KEY[i] else "  "
           for i in range(first)]
    print("correct? :", " ".join(hit))

def report(label, marks):
    r = adjudicate_sheet(key=KEY, marks=marks, profile="balanced")
    print(f"{label}")
    print(f"  verdict          : {r.verdict}")
    print(f"  score as marked  : {r.raw_score} / {N}")
    print(f"  score after      : {r.adjudicated_score} / {N}")
    print(f"  Monte Carlo p    : {r.calibration['p_value']:.4f}")
    return r


## Sheet A — aligned sheet

This sheet has no mechanical error, and the rows line up with the questions.

Expected result: the detector leaves the score unchanged.


In [ ]:
show(WELL)


In [ ]:
a = report("Sheet A — clean", WELL)


The score is unchanged because no shifted alignment is supported by the data.


## Sheet B — one bubble row skipped

This uses the same candidate responses, but leaves question 12 blank and
places the remaining responses one row lower.

The `correct?` markers become sparse after question 11. Some matches still
occur by chance, so a simple count of matching answers is not sufficient.


In [ ]:
SHIFTED = WELL[:11] + [None] + WELL[11:N - 1]
show(SHIFTED)


In [ ]:
b = report("Sheet B — skipped row", SHIFTED)


In [ ]:
print(f"marks recovered: {b.adjudicated_score - b.raw_score}")
print()
print("the boundary where the detector starts trusting the shift:")
for row in b.item_ledger[18:26]:
    print(f"  Q{row['question']:>2}  confidence {row['map_posterior']:.3f}  "
          f"{row['change']:<5} {row['reason'][:42]}")


In this generated example, sixteen marks are recovered. The item ledger
shows where the detector begins to accept the shifted alignment. It requires
at least 99% posterior confidence for the relevant boundary; items below
that threshold remain unchanged.


## Sheet C — random responses

This sheet has no mechanical error, but its responses are random guesses.

It is included as a false-positive check: some alignments can improve a
score by chance, even when no shift occurred.


In [ ]:
GUESSED = [rng.choice(OPTIONS) for _ in range(N)]
c = report("Sheet C — guessing, no error", GUESSED)


The shifted alignment is rejected and the score remains unchanged. For
comparison, the next cell calculates a permissive longest-common-subsequence
score:


In [ ]:
def lcs(a, b):
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0] * (len(b) + 1)
        for j, y in enumerate(b, 1):
            cur[j] = prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1])
        prev = cur
    return prev[-1]

print(f"guessed sheet, marked strictly    : {c.raw_score} / {N}")
print(f"guessed sheet, generous alignment : {lcs(GUESSED, KEY)} / {N}")
print(f"guessed sheet, this detector      : {c.adjudicated_score} / {N}")


The comparison illustrates why score improvement alone is not evidence of a
mechanical shift. The detector therefore uses a stricter acceptance rule.


## The three sheets together


In [ ]:
rows = [("A  clean",            a), ("B  skipped row", b), ("C  guessing", c)]
print(f"{'sheet':<16}{'verdict':<10}{'before':>8}{'after':>7}{'change':>8}")
print('-' * 49)
for name, r in rows:
    v = 'corrected' if r.accepted else 'left as is'
    d = r.adjudicated_score - r.raw_score
    print(f"{name:<16}{v:<10}{r.raw_score:>8}{r.adjudicated_score:>7}{d:>+8}")


One sheet corrected, two left alone. The corrected one is the only one that
had a mechanical error.


## Decision details

Each verdict includes the value produced by each check, the threshold used,
the three null models, and an item-level explanation of the result.


In [ ]:
print(b.explain())


## Try your own sheet

Edit `my_marks` below. Use `None` for a row left blank. Setting `profile` to
`"sensitive"` accepts weaker evidence; `"conservative"` demands more.


In [ ]:
my_marks = WELL[:20] + [None] + WELL[20:N - 1]     # a skip at question 21

mine = adjudicate_sheet(key=KEY, marks=my_marks, profile="balanced")
print(mine.summary())


## What this does not show

Every sheet above is synthetic and selected for illustration. Measurements
from nearly ten thousand additional synthetic sheets are available in the
repository. They show that the detector stays silent on roughly nine of ten
genuine shifts in order to keep false corrections near zero.

| | |
|---|---|
| `README.md` | headline results and limitations |
| `REPORT.md` | the technical report |
| `ASSUMPTIONS.md` | what was assumed, and what measurement said |
| `REPRODUCE.md` | commands that regenerate every published number |
